In [1]:
# PHASE 0: LOAD REAL TWITTER DATASET

import pandas as pd

# Load dataset
df = pd.read_csv("Tweets.csv")

# Keep only relevant columns
df = df[['text', 'airline_sentiment']]

# Convert labels to binary sentiment
df['label'] = df['airline_sentiment'].map({
    'positive': 1,
    'negative': 0
})

# Remove neutral for binary classification
df = df[df['label'].notnull()]

print(df.head())


                                                text airline_sentiment  label
1  @VirginAmerica plus you've added commercials t...          positive    1.0
3  @VirginAmerica it's really aggressive to blast...          negative    0.0
4  @VirginAmerica and it's a really big bad thing...          negative    0.0
5  @VirginAmerica seriously would pay $30 a fligh...          negative    0.0
6  @VirginAmerica yes, nearly every time I fly VX...          positive    1.0


In [9]:
pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [3]:
import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|@\w+|#\w+|[^a-z\s]", "", text)
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return " ".join(words)

df['clean_text'] = df['text'].apply(preprocess_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\saiku\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df['clean_text']
y = df['label']

vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

tfidf_model = LogisticRegression()
tfidf_model.fit(X_train, y_train)

preds = tfidf_model.predict(X_test)
print("TF-IDF Accuracy:", accuracy_score(y_test, preds))

import pickle

# Save TF-IDF vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# Save TF-IDF Logistic Regression model
with open("tfidf_model.pkl", "wb") as f:
    pickle.dump(tfidf_model, f)


TF-IDF Accuracy: 0.9012559549588567


In [5]:
from gensim.models import Word2Vec
import numpy as np

sentences = [text.split() for text in df['clean_text']]

w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

def sentence_vector(sentence):
    words = sentence.split()
    vectors = [w2v_model.wv[w] for w in words if w in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

X_w2v = np.array([sentence_vector(text) for text in df['clean_text']])

X_train, X_test, y_train, y_test = train_test_split(
    X_w2v, y, test_size=0.2, random_state=42
)

w2v_classifier = LogisticRegression()
w2v_classifier.fit(X_train, y_train)

preds = w2v_classifier.predict(X_test)
print("Word2Vec Accuracy:", accuracy_score(y_test, preds))

# Save Word2Vec model
w2v_model.save("word2vec.model")

# Save Word2Vec classifier
with open("word2vec_classifier.pkl", "wb") as f:
    pickle.dump(w2v_classifier, f)



Word2Vec Accuracy: 0.8575140753572975


In [ ]:
pip instakll

In [6]:
# PHASE 5: BERT SENTIMENT ANALYSIS

import torch
from transformers import pipeline

bert_model = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

sample_text = df['text'].iloc[0]
result = bert_model(sample_text)

print(result)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

C:\Users\saiku\OneDrive\Desktop\ML Course\heart-disease-project\env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\saiku\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.6070863008499146}]
